## 🎯 Learning Objectives
* Understand the fundamental concept of Decision Trees as a supervised learning algorithm.
* Explain how Decision Trees make decisions through recursive splitting based on feature values.
* Identify key metrics used for splitting nodes, such as Gini impurity and entropy.
* Analyze the impact of tree depth on model complexity, bias, and variance.
* Apply pre-pruning techniques (e.g., max_depth) to control Decision Tree overfitting.
* Interpret a visualized Decision Tree and understand its decision-making process.


## ML03-L01: Decision Trees: Splitting, Depth, and Pruning

Welcome to the foundational world of Decision Trees! These algorithms are intuitive, powerful, and form the basis for more complex ensemble methods like Random Forests and Gradient Boosting Machines. Imagine playing a game of "20 Questions" or following a flowchart to make a decision – that's essentially how a Decision Tree operates.

### What is a Decision Tree?

AAt its core, a Decision Tree is a non-parametric supervised learning algorithm used for both classification and regression tasks. It works by recursively splitting the dataset into smaller and smaller subsets based on feature values, creating a tree-like structure of decisions. Each internal node represents a "test" on an attribute (e.g., "Is the temperature > 25°C?"), each branch represents the outcome of the test, and each leaf node represents a class label (for classification) or a numerical value (for regression).

**Analogy: Deciding to Play Tennis**

Consider deciding whether to play tennis. You might follow these steps:

1.  **Is the outlook sunny?**
    *   If yes: **Is the humidity high?**
        *   If yes: No tennis.
        *   If no: Play tennis.
    *   If overcast: Play tennis.
    *   If rainy: **Is it windy?**
        *   If yes: No tennis.
        *   If no: Play tennis.

This flowchart is a perfect representation of a Decision Tree!

### Key Concepts:

1.  **Splitting**: The process of dividing a node into two or more sub-nodes. The goal is to create splits that result in the purest possible child nodes, meaning each child node contains data points predominantly belonging to a single class. How do we choose the "best" split?
    *   **Gini Impurity**: Measures the probability of incorrectly classifying a randomly chosen element in the dataset if it were randomly labeled according to the distribution of labels in the subset. A Gini impurity of 0 means all elements belong to a single class (pure node).
    *   **Entropy**: Measures the disorder or uncertainty in a set of examples. An entropy of 0 means the set is perfectly pure. Information Gain is the reduction in entropy achieved by a split.
    *   The algorithm evaluates different features and their thresholds to find the split that maximizes information gain or minimizes Gini impurity.

2.  **Depth**: This refers to the length of the longest path from the root node to a leaf node. A deeper tree means more splits and more complex decision rules. While a deeper tree can capture more intricate patterns in the data, it also increases the risk of **overfitting** – where the model learns the training data too well, including its noise, and performs poorly on unseen data.

3.  **Pruning**: The process of reducing the size of the Decision Tree by removing sections of the tree that provide little power to classify instances. Pruning helps to prevent overfitting and improve the model's generalization ability. There are two main types:
    *   **Pre-pruning (or Early Stopping)**: Stopping the tree construction early. This involves setting limits on tree growth *before* it's fully built (e.g., `max_depth`, `min_samples_leaf`, `min_impurity_decrease`).
    *   **Post-pruning**: Building the full tree first, and then removing branches that have low predictive power or contribute to overfitting. This is often done by evaluating the tree's performance on a validation set.

In this lesson, we'll focus on understanding splitting criteria and demonstrating the impact of tree depth and pre-pruning using `scikit-learn`.


In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Generate a synthetic dataset
# We'll create a simple binary classification dataset with 2 features
# This makes it easy to visualize the decision boundaries.
print("Generating synthetic dataset...")
X, y = make_classification(
    n_samples=300,        # Total number of samples
    n_features=2,         # Number of features
    n_informative=2,      # Number of informative features (all are informative here)
    n_redundant=0,        # Number of redundant features
    n_repeated=0,         # Number of repeated features
    n_classes=2,          # Number of target classes
    n_clusters_per_class=1, # Number of clusters per class
    random_state=42       # For reproducibility
)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Dataset generated: {X_train.shape[0]} training samples, {X_test.shape[0]} testing samples.")

# 2. Train a Decision Tree Classifier without depth limit (prone to overfitting)
print("\nTraining a Decision Tree without depth limit...")
tree_unlimited = DecisionTreeClassifier(random_state=42)
tree_unlimited.fit(X_train, y_train)

y_pred_unlimited = tree_unlimited.predict(X_test)
accuracy_unlimited = accuracy_score(y_test, y_pred_unlimited)
print(f"Accuracy (unlimited depth): {accuracy_unlimited:.4f}")
print(f"Tree depth (unlimited): {tree_unlimited.get_depth()}")

# 3. Train a Decision Tree Classifier with a limited depth (pre-pruning)
# max_depth is a crucial pre-pruning parameter.
print("\nTraining a Decision Tree with max_depth=3...")
tree_limited = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_limited.fit(X_train, y_train)

y_pred_limited = tree_limited.predict(X_test)
accuracy_limited = accuracy_score(y_test, y_pred_limited)
print(f"Accuracy (max_depth=3): {accuracy_limited:.4f}")
print(f"Tree depth (max_depth=3): {tree_limited.get_depth()}")

# 4. Visualize the Decision Trees
# This helps in understanding the splitting process and the impact of depth.
print("\nVisualizing Decision Trees...")

plt.figure(figsize=(18, 10))
plot_tree(
    tree_unlimited, 
    filled=True, 
    feature_names=['Feature 1', 'Feature 2'], 
    class_names=['Class 0', 'Class 1'], 
    rounded=True, 
    fontsize=8
)
plt.title('Decision Tree (Unlimited Depth)', fontsize=16)
plt.show()

plt.figure(figsize=(18, 10))
plot_tree(
    tree_limited, 
    filled=True, 
    feature_names=['Feature 1', 'Feature 2'], 
    class_names=['Class 0', 'Class 1'], 
    rounded=True, 
    fontsize=8
)
plt.title('Decision Tree (Max Depth = 3)', fontsize=16)
plt.show()

# 5. Visualize Decision Boundaries (optional, but highly illustrative)
print("\nVisualizing Decision Boundaries...")

def plot_decision_boundary(clf, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', marker='o')
    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

plt.figure(figsize=(12, 6))
plot_decision_boundary(tree_unlimited, X_test, y_test, 'Decision Boundary (Unlimited Depth)')

plt.figure(figsize=(12, 6))
plot_decision_boundary(tree_limited, X_test, y_test, 'Decision Boundary (Max Depth = 3)')

# 6. Feature Importances
# Decision Trees can also provide insights into feature importance.
print("\nFeature Importances (Unlimited Depth Tree):")
for i, importance in enumerate(tree_unlimited.feature_importances_):
    print(f"Feature {i+1}: {importance:.4f}")

print("\nFeature Importances (Max Depth = 3 Tree):")
for i, importance in enumerate(tree_limited.feature_importances_):
    print(f"Feature {i+1}: {importance:.4f}")


### Interpreting the Output and Performance Trade-offs

From the code execution and visualizations, we can observe several key aspects of Decision Trees:

1.  **Tree Visualization (`plot_tree`)**: Each box (node) in the tree visualization provides crucial information:
    *   **`feature <= threshold`**: This is the splitting rule. For example, `Feature 1 <= 0.5` means if the value of Feature 1 is less than or equal to 0.5, go left; otherwise, go right.
    *   **`gini`**: The Gini impurity of the samples at that node. A lower Gini value indicates a purer node. A leaf node (final decision) will ideally have a Gini of 0.
    *   **`samples`**: The number of training samples that reached this node.
    *   **`value`**: The count of samples per class at this node. For example, `value = [100, 50]` means 100 samples belong to Class 0 and 50 to Class 1.
    *   **`class`**: The majority class in that node. This is the prediction if this node were a leaf node.

2.  **Impact of `max_depth` (Pre-pruning)**:
    *   **Unlimited Depth Tree**: Notice how the tree without `max_depth` can grow very deep, creating complex, jagged decision boundaries. While it might achieve very high accuracy on the *training* data (low bias), it often **overfits** to the noise in the training data, leading to lower accuracy on unseen *test* data (high variance). The decision boundaries are highly specific to the training points.
    *   **Limited Depth Tree (`max_depth=3`)**: By setting `max_depth`, we explicitly limit the complexity. The tree is simpler, easier to interpret, and its decision boundaries are smoother. This acts as a **pre-pruning** technique, preventing the tree from growing too deep and capturing noise. While it might have slightly lower training accuracy (higher bias), it often generalizes better to new data, resulting in higher test accuracy (lower variance).

3.  **Decision Boundaries**: The plots clearly illustrate the difference. The unlimited depth tree creates highly irregular boundaries, trying to perfectly separate every single training point. The limited depth tree creates simpler, more generalized boundaries, which are often more robust.

4.  **Feature Importances**: Decision Trees inherently provide a measure of feature importance. Features that are used in splits closer to the root node, and those that result in significant impurity reduction, are considered more important. This can be a valuable tool for feature selection and understanding which attributes drive the model's decisions.

### Performance Trade-offs:

*   **Bias-Variance Trade-off**: Decision Trees are a prime example of this. A very deep tree has low bias (can model complex relationships) but high variance (sensitive to training data changes, overfits). A shallow tree has high bias (simplistic model) but low variance (more stable). Pruning (like setting `max_depth`) helps find a balance.
*   **Interpretability vs. Accuracy**: Shallow trees are highly interpretable, making them excellent for explaining decisions. Deep trees, while potentially more accurate on training data, become black boxes.
*   **Computational Cost**: Training a very deep tree can be computationally expensive, especially with large datasets. Pruning reduces this cost.

### Typical Use Cases:

*   **Interpretability is key**: When you need to explain *why* a decision was made (e.g., medical diagnosis, loan approval).
*   **Feature Selection**: The `feature_importances_` attribute can guide which features are most relevant.
*   **Baseline Models**: Decision Trees are often used as a strong baseline before moving to more complex ensemble methods.
*   **Non-linear Relationships**: They can capture complex non-linear relationships between features and targets without requiring feature engineering like polynomial terms.

### Limitations:

*   **Instability**: Small changes in the data can lead to a completely different tree structure.
*   **Greedy Approach**: The algorithm makes locally optimal decisions at each split, which doesn't guarantee a globally optimal tree.
*   **Bias towards dominant classes**: If some classes are much more frequent, the tree might be biased towards them. Techniques like `class_weight` can mitigate this.

Understanding these concepts is crucial for effectively using Decision Trees and for building a strong foundation for more advanced tree-based algorithms.


### Resources for Further Learning

*   **Scikit-learn Decision Tree Documentation**: The official documentation is an excellent resource for understanding parameters, methods, and examples. It's always up-to-date with the latest implementations.
    *   [Scikit-learn DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
    *   [Scikit-learn User Guide: Decision Trees](https://scikit-learn.org/stable/modules/tree.html)

*   **Google AI Blog & Research**: Google often publishes accessible articles and research papers on fundamental ML concepts and their applications. Searching for "Decision Trees Google AI" can yield valuable insights.
    *   [Google AI Blog](https://ai.googleblog.com/)

*   **Hugging Face Ecosystem**: While Hugging Face is primarily known for its contributions to Natural Language Processing (NLP) and Transformers, its broader ecosystem and community discussions often touch upon foundational ML concepts and best practices relevant to all AI engineers.
    *   [Hugging Face Blog](https://huggingface.co/blog)

*   **Machine Learning Textbooks**: For a deeper theoretical dive, consider standard machine learning textbooks like "An Introduction to Statistical Learning" (ISLR) or "Elements of Statistical Learning" (ESL), which cover Decision Trees extensively.
